In [31]:
import pandas as pd
import spacy
import re
from tqdm import tqdm
from fuzzywuzzy import fuzz
from fuzzywuzzy import process

In [32]:
nlp = spacy.load("en_core_web_md")


In [33]:
file_path = 'DA - Task 1..xlsx'

In [34]:
data_df = pd.read_excel(file_path, sheet_name='Task')
taxonomy_df = pd.read_excel(file_path, sheet_name='Taxonomy')

In [35]:
# Clean text
def clean_text(text):
    if pd.isna(text):
        return ''
    text = str(text).lower().strip()
    return re.sub(r'\s+', ' ', text)

In [36]:
# Prepare taxonomy categories
def get_clean_taxonomy(col_name):
    return [clean_text(item) for item in taxonomy_df[col_name].dropna().unique()]


In [37]:
categories = {
    'Root Cause': get_clean_taxonomy('Root Cause'),
    'Symptom_Condition': get_clean_taxonomy('Symptom Condition '),
    'Symptom_Component': get_clean_taxonomy('Symptom Component'),
    'Fix_Condition': get_clean_taxonomy('Fix Condition'),
    'Fix_Component': get_clean_taxonomy('Fix Component'),
}


In [38]:
# Cache taxonomy docs for performance
category_docs = {cat: [nlp(text) for text in items] for cat, items in categories.items()}


In [39]:
# Function to get best match using semantic similarity
def get_best_match(input_text, docs, original_texts, threshold=0.75):
    if not input_text:
        return ''
    
    input_doc = nlp(input_text)
    best_score = 0
    best_match = ''
    
    for i, doc in enumerate(docs):
        score = input_doc.similarity(doc)
        if score > best_score:
            best_score = score
            best_match = original_texts[i]
    
    return best_match if best_score >= threshold else ''

In [40]:
# Tagging
tagged_rows = []

print("🔁 Processing rows...")
for _, row in tqdm(data_df.iterrows(), total=len(data_df)):
    complaint = clean_text(row.get('Complaint', ''))
    cause = clean_text(row.get('Cause', ''))
    correction = clean_text(row.get('Correction', ''))

    tagged_row = {
        'Complaint': row.get('Complaint', ''),
        'Cause': row.get('Cause', ''),
        'Correction': row.get('Correction', ''),
        'Root Cause': get_best_match(cause, category_docs['Root Cause'], categories['Root Cause']),
        'Symptom_Condition': get_best_match(complaint, category_docs['Symptom_Condition'], categories['Symptom_Condition']),
        'Symptom_Component': get_best_match(complaint, category_docs['Symptom_Component'], categories['Symptom_Component']),
        'Fix_Condition': get_best_match(correction, category_docs['Fix_Condition'], categories['Fix_Condition']),
        'Fix_Component': get_best_match(correction, category_docs['Fix_Component'], categories['Fix_Component']),
    }

    tagged_rows.append(tagged_row)



🔁 Processing rows...


  0%|                                                                                           | 0/20 [00:00<?, ?it/s]C:\Users\lavanya_kosgi\AppData\Local\Temp\ipykernel_74596\516609510.py:11: UserWarning: [W008] Evaluating Doc.similarity based on empty vectors.
  score = input_doc.similarity(doc)
100%|██████████████████████████████████████████████████████████████████████████████████| 20/20 [00:01<00:00, 11.12it/s]


In [41]:
# Export results
tagged_df = pd.DataFrame(tagged_rows)

In [42]:
tagged_df

,Complaint,Cause,Correction,Root Cause,Symptom_Condition,Symptom_Component,Fix_Condition,Fix_Component
0,VISIBLY NOTICE fasteners under cab on P clips ...,Not tighten at factory.,"GO THROUGH AND RE-TIGHTEN ALL P CLIPS, NUTS, A...",not tighten,won't stay open,,not mentioned,not mentioned
1,Fuel door will not stay open,GAS STRUT NOT INSTALLED OR ANYWHERE ON MACHINE,FOUND GAS STRUT NOT INSTALLED OR ANYWHERE ON M...,not installed,won't stay open,,,gas strut
2,"Compressor pressure line, braided steel, crushed","Compressor pressure line, braided steel, crush...",DRAIN AIR FROM SYSTEM.REMOVE ASSOCIATED P CLIP...,out of fitting,,compressor pressure line,,compressor line
3,Oil running from bottom of machine,OIL RETURN UNDER MACHINE SWIVEL FITTING LEFT L...,OIL RETURN UNDER MACHINE SWIVEL FITTING LEFT L...,out of fitting,oil running,,,
4,MISSING VECTOR & INTRIP UNLOCKS.,MISSING VECTOR & INTRIP UNLOCKS WERE NOT INSTA...,INSTALLED MISSING UNLOCKS RAN AND TESTED.,not included,won't stay open,,cleaned out,not mentioned
5,OIL DRIPPING FROM COUPLER OF RETURN LINE TO PR...,Coupler was leaking.,REMOVE COUPLER FREE UP WITH HAMMER AND SOCKET ...,not included,,compressor pressure line,,
6,COMPONENTS MISSING ON BOOM TO MOUNT SMV SIGN,NOT INCLUDED FROM FACTORY,SERVICE CALL 7 MILES.INSTALL MISSING BRACKETS ...,not included,,,,
7,OIL DRIPPING FROM BOTTOM OF MACHINEPICTURES IN...,O-RING ON MALE QUICK CONNECT STICKING OUT OF F...,SERVICE CALL14 MILES.O-RING ON MALE QUICK CONN...,out of fitting,oil running,,not mentioned,not mentioned
8,OIL LEAK,BLOWN ORING,LOADED TRUCK AND DROVE OUT TO SPRAYER GOT TO S...,no oring,oil leak,,,
9,HARNESS BROKE,POOR MATERIAL IN HARNESS,I WAS OUT TO FARM AND I REPLACED THE NCV HARNE...,poor material,,harness,,


In [43]:
tagged_df.to_excel('Tagged_Task1_Output_Refined.xlsx', index=False)